# Landmark Detection System

Real-time facial, pose, and hand landmark detection using Google MediaPipe. Detects 468 face landmarks, 33 pose landmarks, and 21 hand landmarks per hand.

In [ ]:
# Install required packages (run once)
# !pip install mediapipe opencv-python matplotlib numpy

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import mediapipe as mp
import os

print('MediaPipe version:', mp.__name__)
print('OpenCV version:', cv2.__version__)

## 1. Initialize MediaPipe Solutions

In [ ]:
mp_face_mesh = mp.solutions.face_mesh
mp_pose = mp.solutions.pose
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

# Initialize detectors
face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=False,
    max_num_faces=2,
    refine_landmarks=True,
    min_detection_confidence=0.5
)

pose = mp_pose.Pose(
    static_image_mode=False,
    model_complexity=1,
    min_detection_confidence=0.5
)

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.5
)

print('All detectors initialized.')
print(f'  Face Mesh: 468 landmarks per face')
print(f'  Pose: 33 landmarks')
print(f'  Hands: 21 landmarks per hand')

## 2. Helper Functions

In [ ]:
def detect_landmarks(image_rgb):
    """Run all landmark detectors on an RGB image."""
    results = {}
    results['face'] = face_mesh.process(image_rgb)
    results['pose'] = pose.process(image_rgb)
    results['hands'] = hands.process(image_rgb)
    return results


def draw_all_landmarks(image, results):
    """Draw all detected landmarks on the image."""
    annotated = image.copy()
    
    # Draw face mesh
    if results['face'].multi_face_landmarks:
        for face_landmarks in results['face'].multi_face_landmarks:
            mp_drawing.draw_landmarks(
                image=annotated,
                landmark_list=face_landmarks,
                connections=mp_face_mesh.FACEMESH_TESSELATION,
                landmark_drawing_spec=None,
                connection_drawing_spec=mp_drawing_styles
                    .get_default_face_mesh_tesselation_style()
            )
            mp_drawing.draw_landmarks(
                image=annotated,
                landmark_list=face_landmarks,
                connections=mp_face_mesh.FACEMESH_CONTOURS,
                landmark_drawing_spec=None,
                connection_drawing_spec=mp_drawing_styles
                    .get_default_face_mesh_contours_style()
            )
    
    # Draw pose landmarks
    if results['pose'].pose_landmarks:
        mp_drawing.draw_landmarks(
            image=annotated,
            landmark_list=results['pose'].pose_landmarks,
            connections=mp_pose.POSE_CONNECTIONS,
            landmark_drawing_spec=mp_drawing_styles
                .get_default_pose_landmarks_style()
        )
    
    # Draw hand landmarks
    if results['hands'].multi_hand_landmarks:
        for hand_landmarks in results['hands'].multi_hand_landmarks:
            mp_drawing.draw_landmarks(
                image=annotated,
                landmark_list=hand_landmarks,
                connections=mp_hands.HAND_CONNECTIONS,
                landmark_drawing_spec=None,
                connection_drawing_spec=mp_drawing_styles
                    .get_default_hand_connections_style()
            )
    
    return annotated


def count_landmarks(results):
    """Count detected landmarks."""
    counts = {}
    counts['faces'] = len(results['face'].multi_face_landmarks) if results['face'].multi_face_landmarks else 0
    counts['pose'] = 1 if results['pose'].pose_landmarks else 0
    counts['hands'] = len(results['hands'].multi_hand_landmarks) if results['hands'].multi_hand_landmarks else 0
    return counts


def display_image(image, figsize=(10, 8)):
    """Display an image with matplotlib."""
    if len(image.shape) == 3 and image.shape[2] == 3:
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    else:
        image_rgb = image
    plt.figure(figsize=figsize)
    plt.imshow(image_rgb)
    plt.axis('off')
    plt.show()

## 3. Download Sample Images

In [ ]:
import urllib.request

sample_urls = {
    'person.jpg': 'https://images.unsplash.com/photo-1507003211169-0a1dd7228f2d?w=800',
    'group.jpg': 'https://images.unsplash.com/photo-1529156069898-49953e39b3ac?w=800',
}

for name, url in sample_urls.items():
    if not os.path.exists(name):
        print(f'Downloading {name}...')
        urllib.request.urlretrieve(url, name)
        print(f'  Saved {name}')
    else:
        print(f'{name} already exists')

## 4. Run Landmark Detection on an Image

Process any image and visualize all detected landmarks.

In [ ]:
# === CHANGE THIS PATH ===
IMAGE_PATH = 'person.jpg'
# =========================

if not os.path.exists(IMAGE_PATH):
    print(f'Image not found: {IMAGE_PATH}')
else:
    image = cv2.imread(IMAGE_PATH)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    results = detect_landmarks(image_rgb)
    annotated = draw_all_landmarks(image, results)
    
    print('Detected landmarks:')
    counts = count_landmarks(results)
    for k, v in counts.items():
        print(f'  {k}: {v}')
    
    display_image(annotated)

## 5. Face Landmark Details

Extract specific facial landmark coordinates (eyes, nose, mouth, etc.).

In [ ]:
# Key facial feature landmark indices (MediaPipe Face Mesh)
FACE_INDICES = {
    'Left eye (center)': 468,
    'Right eye (center)': 473,
    'Nose tip': 1,
    'Mouth center': 13,
    'Left ear': 234,
    'Right ear': 454,
    'Chin': 152,
    'Left eyebrow': 282,
    'Right eyebrow': 52,
}


def extract_face_landmarks(image_rgb):
    """Extract specific facial landmark coordinates."""
    h, w = image_rgb.shape[:2]
    result = face_mesh.process(image_rgb)
    
    if not result.multi_face_landmarks:
        print('No face detected.')
        return None
    
    landmarks = result.multi_face_landmarks[0]
    points = {}
    for name, idx in FACE_INDICES.items():
        lm = landmarks.landmark[idx]
        x, y = int(lm.x * w), int(lm.y * h)
        points[name] = (x, y)
    
    return points


def visualize_face_landmarks(image, points):
    """Draw specific facial landmarks with labels."""
    annotated = image.copy()
    for name, (x, y) in points.items():
        cv2.circle(annotated, (x, y), 3, (0, 255, 0), -1)
        cv2.putText(annotated, name, (x + 5, y - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
    return annotated


if os.path.exists('person.jpg'):
    image = cv2.imread('person.jpg')
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    points = extract_face_landmarks(image_rgb)
    if points:
        annotated = visualize_face_landmarks(image, points)
        display_image(annotated)
        print('Key landmarks:')
        for name, (x, y) in points.items():
            print(f'  {name:<25} ({x}, {y})')
else:
    print('Download sample images first (Section 3).')

## 6. Pose Landmarks with Angles

Calculate joint angles from pose landmarks (useful for exercise/ergonomics).

In [ ]:
def calculate_angle(a, b, c):
    """Calculate angle ABC (in degrees) given three points."""
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)
    ba = a - b
    bc = c - b
    cos_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc))
    angle = np.degrees(np.arccos(np.clip(cos_angle, -1.0, 1.0)))
    return angle


def analyze_pose(image_rgb):
    """Analyze pose and return joint angles."""
    h, w = image_rgb.shape[:2]
    result = pose.process(image_rgb)
    
    if not result.pose_landmarks:
        print('No pose detected.')
        return None, None
    
    lm = result.pose_landmarks.landmark
    
    # Helper to get (x, y) pixel coords
    def p(idx):
        return (int(lm[idx].x * w), int(lm[idx].y * h))
    
    angles = {
        'Left elbow': calculate_angle(p(11), p(13), p(15)),
        'Right elbow': calculate_angle(p(12), p(14), p(16)),
        'Left shoulder': calculate_angle(p(13), p(11), p(23)),
        'Right shoulder': calculate_angle(p(14), p(12), p(24)),
        'Left knee': calculate_angle(p(23), p(25), p(27)),
        'Right knee': calculate_angle(p(24), p(26), p(28)),
    }
    
    return result.pose_landmarks, angles


if os.path.exists('person.jpg'):
    image = cv2.imread('person.jpg')
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    landmarks, angles = analyze_pose(image_rgb)
    if angles:
        print('Joint angles (degrees):')
        for joint, angle in angles.items():
            print(f'  {joint:<20} {angle:.1f}')
else:
    print('Download sample images first.')

## 7. Hand Landmark Details

Detect hand landmarks and recognize left vs right hand.

In [ ]:
HAND_LANDMARK_NAMES = [
    'WRIST', 'THUMB_CMC', 'THUMB_MCP', 'THUMB_IP', 'THUMB_TIP',
    'INDEX_FINGER_MCP', 'INDEX_FINGER_PIP', 'INDEX_FINGER_DIP', 'INDEX_FINGER_TIP',
    'MIDDLE_FINGER_MCP', 'MIDDLE_FINGER_PIP', 'MIDDLE_FINGER_DIP', 'MIDDLE_FINGER_TIP',
    'RING_FINGER_MCP', 'RING_FINGER_PIP', 'RING_FINGER_DIP', 'RING_FINGER_TIP',
    'PINKY_MCP', 'PINKY_PIP', 'PINKY_DIP', 'PINKY_TIP'
]


def analyze_hands(image_rgb):
    """Detect and label hands with landmark coordinates."""
    h, w = image_rgb.shape[:2]
    result = hands.process(image_rgb)
    
    if not result.multi_hand_landmarks:
        print('No hands detected.')
        return None
    
    hand_data = []
    for hand_lm, handedness in zip(result.multi_hand_landmarks, result.multi_handedness):
        label = handedness.classification[0].label  # 'Left' or 'Right'
        points = {}
        for i, name in enumerate(HAND_LANDMARK_NAMES):
            lm = hand_lm.landmark[i]
            points[name] = (int(lm.x * w), int(lm.y * h))
        hand_data.append({'label': label, 'points': points})
    
    return hand_data


if os.path.exists('person.jpg'):
    image = cv2.imread('person.jpg')
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    hand_data = analyze_hands(image_rgb)
    if hand_data:
        for hand in hand_data:
            print(f'{hand["label"]} hand:')
            for name, (x, y) in hand['points'].items():
                if name.endswith('_TIP'):
                    print(f'  {name:<20} ({x}, {y})')
else:
    print('Download sample images first.')

## 8. Real-time Webcam Landmark Detection (Local Jupyter)

⚠ Requires local Jupyter kernel. Press 'q' to quit.

In [ ]:
def webcam_landmarks(camera_id=0):
    """Real-time face, pose, and hand landmark detection via webcam."""
    cap = cv2.VideoCapture(camera_id)
    if not cap.isOpened():
        print('Cannot open camera.')
        return
    
    print('Webcam started. Press q to quit.')
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = detect_landmarks(frame_rgb)
        annotated = draw_all_landmarks(frame, results)
        
        # Show counts
        counts = count_landmarks(results)
        info = f'Faces: {counts["faces"]} | Pose: {counts["pose"]} | Hands: {counts["hands"]}'
        cv2.putText(annotated, info, (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        
        cv2.imshow('Landmark Detection (q to quit)', annotated)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    cap.release()
    cv2.destroyAllWindows()
    
    # Cleanup MediaPipe resources
    face_mesh.close()
    pose.close()
    hands.close()
    print('Webcam stopped.')

# Uncomment to run:
# webcam_landmarks()

## 9. Batch Processing

Process all images in a folder and save annotated results.

In [ ]:
def batch_process_landmarks(input_folder, output_folder='annotated'):
    """Run landmark detection on all images in a folder."""
    os.makedirs(output_folder, exist_ok=True)
    
    valid_exts = ('.jpg', '.jpeg', '.png', '.bmp')
    images = [f for f in os.listdir(input_folder) if f.lower().endswith(valid_exts)]
    
    if not images:
        print(f'No images found in {input_folder}')
        return
    
    print(f'Processing {len(images)} images...')
    for img_name in images:
        path = os.path.join(input_folder, img_name)
        image = cv2.imread(path)
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        results = detect_landmarks(image_rgb)
        annotated = draw_all_landmarks(image, results)
        
        out_path = os.path.join(output_folder, img_name)
        cv2.imwrite(out_path, annotated)
    
    print(f'Done! Results in {output_folder}/')

# Example: batch_process_landmarks('my_photos')

## 10. Process a Video File

Run landmark detection on every frame of a video.

In [ ]:
def process_video_landmarks(video_path, output_path='annotated_video.mp4'):
    """Apply landmark detection to a video file."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f'Cannot open: {video_path}')
        return
    
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    print(f'Processing video: {total} frames')
    count = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = detect_landmarks(frame_rgb)
        annotated = draw_all_landmarks(frame, results)
        out.write(annotated)
        
        count += 1
        if count % 30 == 0:
            print(f'  {count}/{total}')
    
    cap.release()
    out.release()
    print(f'Done! Saved to {output_path}')

# Example: process_video_landmarks('input.mp4', 'output.mp4')